# Colab 17c Training (YOLOv8n-seg + YOLO26n-seg)
Executa este notebook no kernel Colab com GPU (T4/L4/A100).

In [17]:
# 0) Check Colab runtime
import google.colab
print("OK: runtime Colab")

OK: runtime Colab


In [ ]:
# 0.1) Download dataset zip from Google Drive (optional)
!pip -q install gdown
import re, gdown, os, glob

SHARE_URL = "https://drive.google.com/file/d/11YFhSTlZGjxUqh4DE21erDDrCl_t8xHw/view?usp=drive_link"

m = re.search(r"/d/([^/]+)", SHARE_URL)
assert m, "Link inválido. Deve ser link de ficheiro do Google Drive."
file_id = m.group(1)

out_zip = "/content/object labels.yolo26.zip"
gdown.download(id=file_id, output=out_zip, quiet=False)

print("Exists:", os.path.exists(out_zip), out_zip)
print("ZIPs em /content:", glob.glob("/content/*.zip"))

Downloading...
From (original): https://drive.google.com/uc?id=11YFhSTlZGjxUqh4DE21erDDrCl_t8xHw
From (redirected): https://drive.google.com/uc?id=11YFhSTlZGjxUqh4DE21erDDrCl_t8xHw&confirm=t&uuid=96ddc97b-4d5e-4562-aadb-da23a558628c
To: /content/object labels.yolo26.zip
100%|██████████| 211M/211M [00:02<00:00, 91.7MB/s] 

Exists: True /content/object labels.yolo26.zip
ZIPs em /content: ['/content/seg17c_artifacts_bundle.zip', '/content/object labels.yolo26.zip']


In [4]:
# 1) Install dependencies, remount Drive, check GPU
!pip install -q ultralytics pyyaml
import os
import torch
from pathlib import Path

try:
    from google.colab import drive
    # Force remount so you can pick the correct Google account every run.
    drive.mount('/drive', force_remount=True)
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Drive mount skipped or unavailable: [dfs_ephemeral] Credentials propagation unsuccessful
CUDA available: True
GPU: Tesla T4


In [ ]:
# 2) Config + locate/extract dataset zip
import os, zipfile, glob
from pathlib import Path

drive_roots = [
    '/drive/MyDrive',
    '/content/drive/MyDrive',
]
DRIVE_ROOT = next((path for path in drive_roots if Path(path).exists()), None)

if DRIVE_ROOT:
    OUTPUT_DIR = f'{DRIVE_ROOT}/seame_seg_17c'
    print('Using DRIVE_ROOT=', DRIVE_ROOT)
else:
    # Fallback when Drive mount fails in VS Code Colab integration.
    OUTPUT_DIR = '/content/seame_seg_17c'
    print('Google Drive not mounted. Using local runtime storage:', OUTPUT_DIR)

DATASET_DIR = '/content/dataset_17c_raw'
SPLIT_DIR = '/content/dataset_17c'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if DRIVE_ROOT:
    print('Drive root sample:')
    for path in sorted(glob.glob(f'{DRIVE_ROOT}/*'))[:20]:
        print(' -', path)

candidate_zips = [
    '/content/seame_dataset_17c.zip',
    '/content/object labels.yolo26.zip',
]

if DRIVE_ROOT:
    candidate_zips.extend([
        f'{DRIVE_ROOT}/seame_dataset_17c.zip',
        f'{DRIVE_ROOT}/object labels.yolo26.zip',
        f'{DRIVE_ROOT}/Downloads/seame_dataset_17c.zip',
        f'{DRIVE_ROOT}/Downloads/object labels.yolo26.zip',
    ])

search_patterns = [
    '/content/**/*.zip',
]
if DRIVE_ROOT:
    search_patterns.extend([
        f'{DRIVE_ROOT}/**/*17c*.zip',
        f'{DRIVE_ROOT}/**/*dataset*.zip',
        f'{DRIVE_ROOT}/**/*yolo26*.zip',
    ])

for pattern in search_patterns:
    candidate_zips.extend(sorted(glob.glob(pattern, recursive=True)))

seen = set()
candidate_zips = [p for p in candidate_zips if not (p in seen or seen.add(p))]
existing_zips = [p for p in candidate_zips if os.path.exists(p)]

print('Found zip candidates:')
for path in existing_zips[:50]:
    print(' -', path)

assert existing_zips, (
    'No dataset zip found. Upload the zip to /content (runtime Files panel) or fix Drive auth and rerun cell 1.'
)

DATASET_ZIP = existing_zips[0]
print('Using DATASET_ZIP=', DATASET_ZIP)

if not os.path.exists(DATASET_DIR):
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(DATASET_DIR)

candidates = sorted(glob.glob(f'{DATASET_DIR}/**/train/images', recursive=True))
assert candidates, 'train/images not found in extracted dataset'
TRAIN_IMG_SRC = candidates[0]
TRAIN_LBL_SRC = TRAIN_IMG_SRC.replace('/images', '/labels')
print('TRAIN_IMG_SRC=', TRAIN_IMG_SRC)
print('TRAIN_LBL_SRC=', TRAIN_LBL_SRC)

Google Drive not mounted. Using local runtime storage: /content/seame_seg_17c
Found zip candidates:
 - /content/object labels.yolo26.zip
 - /content/seg17c_artifacts_bundle.zip
Using DATASET_ZIP= /content/object labels.yolo26.zip
TRAIN_IMG_SRC= /content/dataset_17c_raw/train/images
TRAIN_LBL_SRC= /content/dataset_17c_raw/train/labels


In [ ]:
# 3) Create 80/20 split, sanitize seg labels, and write data.yaml
import random, shutil, yaml
from pathlib import Path

SEED = 42
random.seed(SEED)

# Common image extensions
all_imgs = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
    all_imgs.extend(Path(TRAIN_IMG_SRC).glob(ext))
all_imgs = sorted(set(all_imgs))
random.shuffle(all_imgs)

def sanitize_seg_label_lines(lbl_path: Path, nc: int = 17):
    """
    Return only valid YOLO-seg polygon lines.
    Valid line = class_id + even number of coords and at least 3 points (>= 7 tokens total).
    Also enforces class range [0, nc-1] and normalized coords in [0, 1].
    """
    if not lbl_path.exists():
        return []

    try:
        raw_lines = [ln.strip() for ln in lbl_path.read_text().splitlines() if ln.strip()]
    except Exception:
        return []

    cleaned = []
    for ln in raw_lines:
        parts = ln.split()
        if len(parts) < 7:
            continue

        try:
            cls_id = int(float(parts[0]))
            coords = [float(x) for x in parts[1:]]
        except Exception:
            continue

        if cls_id < 0 or cls_id >= nc:
            continue
        if len(coords) % 2 != 0:
            continue
        if len(coords) < 6:
            continue
        if any((c < 0.0 or c > 1.0) for c in coords):
            continue

        cleaned.append(str(cls_id) + ' ' + ' '.join(f'{c:.6f}' for c in coords))

    return cleaned

# Filter to images with at least one valid segmentation polygon
valid_pairs = []
invalid_count = 0
for p in all_imgs:
    lp = Path(TRAIN_LBL_SRC) / (p.stem + '.txt')
    cleaned_lines = sanitize_seg_label_lines(lp, nc=17)
    if cleaned_lines:
        valid_pairs.append((p, cleaned_lines))
    else:
        invalid_count += 1

assert valid_pairs, 'No valid image/label pairs found after label sanitization.'
print(f'Total images found: {len(all_imgs)}')
print(f'Valid seg pairs: {len(valid_pairs)}')
print(f'Invalid/empty labels skipped: {invalid_count}')

cut = int(0.8 * len(valid_pairs))
train_pairs, valid_pairs = valid_pairs[:cut], valid_pairs[cut:]
assert train_pairs and valid_pairs, 'Split failed: train/valid is empty after filtering.'

# Rebuild split folders from scratch to avoid stale files from old runs.
for split in ('train', 'valid'):
    split_root = Path(SPLIT_DIR) / split
    if split_root.exists():
        shutil.rmtree(split_root)

for split, pairs in [('train', train_pairs), ('valid', valid_pairs)]:
    img_out = Path(SPLIT_DIR) / split / 'images'
    lbl_out = Path(SPLIT_DIR) / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_p, cleaned_lines in pairs:
        shutil.copy2(img_p, img_out / img_p.name)
        (lbl_out / f'{img_p.stem}.txt').write_text('\n'.join(cleaned_lines) + '\n')

CLASSES_17 = [
    '50_maxspeed','80_maxspeed','Crosswalk','Gate','Pedestrians_crossing','Stop_sign',
    'Traffic_priority','both_arrow','car','cars not allowed','left_cross','obstacle',
    'right_cross','traffic_lights_green','traffic_lights_off','traffic_lights_red','traffic_lights_yellow'
]
DATA_YAML = f'{SPLIT_DIR}/data.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(
        {'path': SPLIT_DIR, 'train': 'train/images', 'val': 'valid/images', 'nc': 17, 'names': CLASSES_17},
        f,
        default_flow_style=False,
    )

# Clear Ultralytics label caches so training always reads fresh sanitized labels.
for cache_file in (
    Path(SPLIT_DIR) / 'train' / 'labels.cache',
    Path(SPLIT_DIR) / 'valid' / 'labels.cache',
):
    if cache_file.exists():
        cache_file.unlink()
        print('Removed cache:', cache_file)

print('DATA_YAML=', DATA_YAML)
print('Train images:', len(list((Path(SPLIT_DIR)/'train'/'images').glob('*'))))
print('Valid images:', len(list((Path(SPLIT_DIR)/'valid'/'images').glob('*'))))

Total images found: 1973
Valid seg pairs: 90
Invalid/empty labels skipped: 1883
DATA_YAML= /content/dataset_17c/data.yaml
Train images: 72
Valid images: 18


In [ ]:
# 3.1) Sanity-check split labels before training
from pathlib import Path

def is_clean_seg_line(ln: str, nc: int = 17) -> bool:
    parts = ln.split()
    if len(parts) < 7:
        return False
    try:
        cls_id = int(parts[0])
        coords = [float(x) for x in parts[1:]]
    except Exception:
        return False
    if cls_id < 0 or cls_id >= nc:
        return False
    if len(coords) < 6 or len(coords) % 2 != 0:
        return False
    if any((c < 0.0 or c > 1.0) for c in coords):
        return False
    return True

bad_files = []
line_count = 0
for split in ('train', 'valid'):
    lbl_dir = Path(SPLIT_DIR) / split / 'labels'
    for p in lbl_dir.glob('*.txt'):
        lines = [ln.strip() for ln in p.read_text().splitlines() if ln.strip()]
        if not lines:
            bad_files.append(str(p))
            continue
        if not all(is_clean_seg_line(ln, nc=17) for ln in lines):
            bad_files.append(str(p))
            continue
        line_count += len(lines)

print('Label files checked:', len(list((Path(SPLIT_DIR)/'train'/'labels').glob('*.txt'))) + len(list((Path(SPLIT_DIR)/'valid'/'labels').glob('*.txt'))))
print('Total polygon lines:', line_count)
print('Invalid label files:', len(bad_files))

assert not bad_files, f'Found invalid labels after sanitization. Example: {bad_files[:3]}'
print('OK: split labels are clean for segmentation training.')

Label files checked: 90
Total polygon lines: 91
Invalid label files: 0
OK: split labels are clean for segmentation training.


In [ ]:
# 3.2) Audit sanitized dataset quality and class coverage
from collections import Counter
from pathlib import Path

split_class_counts = Counter()
split_instance_counts = Counter()
file_stats = {'train': 0, 'valid': 0}

for split in ('train', 'valid'):
    lbl_dir = Path(SPLIT_DIR) / split / 'labels'
    for p in lbl_dir.glob('*.txt'):
        file_stats[split] += 1
        for ln in [ln.strip() for ln in p.read_text().splitlines() if ln.strip()]:
            cls_id = int(ln.split()[0])
            split_class_counts[cls_id] += 1
            split_instance_counts[(split, cls_id)] += 1

print('Sanitized label files by split:', file_stats)
print('Classes present after sanitization:')
for cls_id in sorted(split_class_counts):
    print(f'  {cls_id:02d} {CLASSES_17[cls_id]} -> {split_class_counts[cls_id]} polygons')

missing = [CLASSES_17[i] for i in range(len(CLASSES_17)) if i not in split_class_counts]
print('Missing classes after sanitization:', missing if missing else 'None')

Sanitized label files by split: {'train': 72, 'valid': 18}
Classes present after sanitization:
  03 Gate -> 88 polygons
  11 obstacle -> 2 polygons
  15 traffic_lights_red -> 1 polygons
Missing classes after sanitization: ['50_maxspeed', '80_maxspeed', 'Crosswalk', 'Pedestrians_crossing', 'Stop_sign', 'Traffic_priority', 'both_arrow', 'car', 'cars not allowed', 'left_cross', 'right_cross', 'traffic_lights_green', 'traffic_lights_off', 'traffic_lights_yellow']


In [79]:
# 4) Train yolov8n-seg
from ultralytics import YOLO
EPOCHS = 50; BATCH = 16; IMGSZ = 640
m8 = YOLO('yolov8n-seg.pt')
m8.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    workers=0,
    seed=42,
    deterministic=True,
    project=OUTPUT_DIR,
    name='yolov8n_seg_17c',
    exist_ok=True,
    patience=20,
    amp=True,
    close_mosaic=10,
    optimizer='auto',
    copy_paste=0.0,
    overlap_mask=False,
    mask_ratio=1,
    cache=False,
 )
BEST_8N = f'{OUTPUT_DIR}/yolov8n_seg_17c/weights/best.pt'
print('BEST_8N=', BEST_8N, os.path.exists(BEST_8N))

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_17c/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=1, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_seg_17c, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=False, p

: 

: 

In [ ]:
# 5) Train yolo26n-seg
m26 = YOLO('yolo26n-seg.pt')
m26.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=0,
    workers=0,
    seed=42,
    deterministic=True,
    project=OUTPUT_DIR,
    name='yolo26n_seg_17c',
    exist_ok=True,
    patience=20,
    amp=True,
    close_mosaic=10,
    optimizer='auto',
    copy_paste=0.0,
    overlap_mask=False,
    mask_ratio=1,
    cache=False,
 )
BEST_26N = f'{OUTPUT_DIR}/yolo26n_seg_17c/weights/best.pt'
print('BEST_26N=', BEST_26N, os.path.exists(BEST_26N))

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_17c/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=1, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_seg_17c, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=False, p

In [ ]:
# 6) Validate + optional predict videos
v8 = m8.val(data=DATA_YAML, split='val', imgsz=IMGSZ, device=0, batch=BATCH)
v26 = m26.val(data=DATA_YAML, split='val', imgsz=IMGSZ, device=0, batch=BATCH)
print(f'yolov8n-seg: mAP50={v8.seg.map50:.3f} mAP50-95={v8.seg.map:.3f}')
print(f'yolo26n-seg: mAP50={v26.seg.map50:.3f} mAP50-95={v26.seg.map:.3f}')
# Example predict (replace source with your video/images)
# m26.predict(source='/content/sample.mp4', conf=0.25, iou=0.5, imgsz=640, save=True)

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,261,379 parameters, 0 gradients, 11.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2400.8±537.0 MB/s, size: 75.3 KB)
val: Scanning /content/dataset_17c/valid/labels.cache... 18 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 18/18 5.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.7it/s 0.7s1.2s
                   all         18         19          1      0.472      0.513      0.375          1      0.472      0.513      0.336
                  Gate         18         18          1      0.944      0.995      0.742          1      0.944      0.995      0.657
              obstacle          1          1          1          0     0.0302    0.00905          1          0     0.0302     0.0151
Speed: 6.9ms preprocess, 14.1ms inference,

In [ ]:
# 6.1) Predict on validation images with trained models
from pathlib import Path

PREDICT_SOURCE = f'{SPLIT_DIR}/valid/images'
PREDICT_IMGSZ = IMGSZ

for model_name, best_path in [('yolov8n_seg_17c', BEST_8N), ('yolo26n_seg_17c', BEST_26N)]:
    model = YOLO(best_path)
    model.predict(
        source=PREDICT_SOURCE,
        imgsz=PREDICT_IMGSZ,
        conf=0.25,
        iou=0.5,
        device=0,
        save=True,
        project=OUTPUT_DIR,
        name=f'{model_name}_predict',
        exist_ok=True,
    )
    save_dir = Path(OUTPUT_DIR) / f'{model_name}_predict'
    generated = sorted([p.name for p in save_dir.rglob('*') if p.is_file()])
    print(model_name, 'predict source:', PREDICT_SOURCE)
    print(model_name, 'predict outputs:', save_dir, save_dir.exists(), 'files:', generated[:20])


image 1/18 /content/dataset_17c/valid/images/frame_00096_jpg.rf.y2zaCA0pPxF2lmSmRBrC.jpg: 480x640 (no detections), 51.2ms
image 2/18 /content/dataset_17c/valid/images/frame_00101_jpg.rf.kNdHTOcS0vCqUgjP5WGM.jpg: 480x640 (no detections), 7.5ms
image 3/18 /content/dataset_17c/valid/images/frame_00104_jpg.rf.r3EkC4kIDnZsWmfAUnNZ.jpg: 480x640 (no detections), 7.4ms
image 4/18 /content/dataset_17c/valid/images/frame_00107_jpg.rf.aX2xX1Z9F1wGbWdMsWAF.jpg: 480x640 (no detections), 7.7ms
image 5/18 /content/dataset_17c/valid/images/frame_00111_jpg.rf.cSHm7ZTFIbpXKxW5cu3y.jpg: 480x640 (no detections), 7.4ms
image 6/18 /content/dataset_17c/valid/images/frame_00113_jpg.rf.tqtbrxs8ep1KPHuVIIFb.jpg: 480x640 (no detections), 7.4ms
image 7/18 /content/dataset_17c/valid/images/frame_00130_jpg.rf.k48E5NJ5n0vQn4HdFsu6.jpg: 480x640 (no detections), 7.4ms
image 8/18 /content/dataset_17c/valid/images/frame_00136_jpg.rf.r6BAlIf8eFjjyoaZ25LM.jpg: 480x640 (no detections), 8.4ms
image 9/18 /content/dataset_17

## Execution Order (Processo-Mae Obrigatorio)
Segue esta ordem sem saltar etapas:

1. `# 7.0) Auto-discover trained weight paths`
2. `# 7) Print final files to download`
3. `# 6.1) Predict on validation images with trained models`
4. Comparar com YOLOv8s do Vasco (fora deste notebook, no fluxo local/benchmark).
5. `# 7.1) Bundle trained artifacts into a single zip for download`
6. `# 7.12) Set HF token in runtime (no notebook save)`
7. `# 7.13) Validate Hugging Face token`
8. `# 7.15) Configure HF repo settings (token from runtime env)`
9. `# 7.2) Non-interactive upload to Hugging Face Hub`
10. `# 8) Export trained models to ONNX` (so depois da comparacao com o Vasco).

Notas:
- Em caso de `HF upload failed (HTTP 401)`, renovar token e repetir as Cells 17, 18 e 20.
- Nao fazer reconnect/restart do Colab sem instrucao explicita neste fluxo.
- Se nao correste validacao (`# 6`), `metrics.json` pode sair com valores `null`.

In [19]:
# 7) Print final files to download
import os
import hashlib

if 'BEST_8N' not in globals() or 'BEST_26N' not in globals():
    raise RuntimeError('Missing BEST_8N/BEST_26N. Run the auto-discovery helper cell first.')

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

for p in [BEST_8N, BEST_26N]:
    print('FILE:', p)
    print('SIZE_MB:', round(os.path.getsize(p)/1024/1024,2))
    print('SHA256:', sha256_of(p))
    print('---')

print('Copy to Lenovo:')
print('~/Documents/AI/Yolo_benchmark/trained/yolov8n_seg_17c/weights/best.pt')
print('~/Documents/AI/Yolo_benchmark/trained/yolo26n_seg_17c/weights/best.pt')

FILE: /content/seame_seg_17c/yolov8n_seg_17c/weights/best.pt
SIZE_MB: 6.46
SHA256: 7a864c44073fbdbb0d4058cc966c4d09b648209541a99e73002d52b25d6f68bd
---
FILE: /content/seame_seg_17c/yolo26n_seg_17c/weights/best.pt
SIZE_MB: 6.24
SHA256: 53bceb789252e0bb1658a2199c5a6d549a095753d42d0e627b87eefc15e915a9
---
Copy to Lenovo:
~/Documents/AI/Yolo_benchmark/trained/yolov8n_seg_17c/weights/best.pt
~/Documents/AI/Yolo_benchmark/trained/yolo26n_seg_17c/weights/best.pt


In [18]:
# 7.0) Auto-discover trained weight paths
import glob, os
from pathlib import Path

# tenta descobrir OUTPUT_DIR automaticamente
if "OUTPUT_DIR" not in globals() or not OUTPUT_DIR:
    candidates = glob.glob("/content/**/yolov8n_seg_17c/weights/best.pt", recursive=True)
    if not candidates:
        candidates = glob.glob("/drive/**/yolov8n_seg_17c/weights/best.pt", recursive=True)
    assert candidates, "Nao encontrei best.pt do yolov8n_seg_17c"
    BEST_8N = sorted(candidates)[-1]
    OUTPUT_DIR = str(Path(BEST_8N).parents[2])

BEST_8N = str(Path(OUTPUT_DIR) / "yolov8n_seg_17c/weights/best.pt")
BEST_26N = str(Path(OUTPUT_DIR) / "yolo26n_seg_17c/weights/best.pt")

assert os.path.exists(BEST_8N), f"Falta {BEST_8N}"
assert os.path.exists(BEST_26N), f"Falta {BEST_26N}"

print("OUTPUT_DIR =", OUTPUT_DIR)
print("BEST_8N =", BEST_8N)
print("BEST_26N =", BEST_26N)

OUTPUT_DIR = /content/seame_seg_17c
BEST_8N = /content/seame_seg_17c/yolov8n_seg_17c/weights/best.pt
BEST_26N = /content/seame_seg_17c/yolo26n_seg_17c/weights/best.pt


In [35]:
# 7.1) Bundle trained artifacts into a single zip for download
import json
import shutil
from pathlib import Path

if 'BEST_8N' not in globals() or 'BEST_26N' not in globals():
    raise RuntimeError('Missing BEST_8N/BEST_26N. Run the auto-discovery helper cell first.')

bundle_root = Path('/content/seg17c_artifacts')
if bundle_root.exists():
    shutil.rmtree(bundle_root)
bundle_root.mkdir(parents=True, exist_ok=True)

onnx_8n = str(Path(BEST_8N).with_suffix('.onnx'))
onnx_26n = str(Path(BEST_26N).with_suffix('.onnx'))

def metric_or_none(obj, attr_chain):
    try:
        cur = obj
        for name in attr_chain:
            cur = getattr(cur, name)
        return float(cur)
    except Exception:
        return None

artifacts = {
    'yolov8n_seg_best_pt': BEST_8N,
    'yolo26n_seg_best_pt': BEST_26N,
    'yolov8n_seg_best_onnx': onnx_8n if Path(onnx_8n).exists() else None,
    'yolo26n_seg_best_onnx': onnx_26n if Path(onnx_26n).exists() else None,
    'yolov8n_seg_map50': metric_or_none(globals().get('v8', None), ['seg', 'map50']),
    'yolov8n_seg_map50_95': metric_or_none(globals().get('v8', None), ['seg', 'map']),
    'yolo26n_seg_map50': metric_or_none(globals().get('v26', None), ['seg', 'map50']),
    'yolo26n_seg_map50_95': metric_or_none(globals().get('v26', None), ['seg', 'map']),
}

named_sources = {
    'yolov8n_seg_17c_best.pt': BEST_8N,
    'yolo26n_seg_17c_best.pt': BEST_26N,
}
if Path(onnx_8n).exists():
    named_sources['yolov8n_seg_17c_best.onnx'] = onnx_8n
if Path(onnx_26n).exists():
    named_sources['yolo26n_seg_17c_best.onnx'] = onnx_26n

for dest_name, src in named_sources.items():
    shutil.copy2(src, bundle_root / dest_name)

(bundle_root / 'metrics.json').write_text(json.dumps(artifacts, indent=2))
(bundle_root / 'README.txt').write_text(
    'Artifacts bundled from colab_seg_train_17c.ipynb\n'
    'Files may include PT and ONNX exports when available.\n'
    'Metrics may be null if validation cell was not run in this kernel session.\n'
)

zip_base = '/content/seg17c_artifacts_bundle'
zip_path = shutil.make_archive(zip_base, 'zip', bundle_root)
print('BUNDLE_DIR=', bundle_root)
print('BUNDLE_ZIP=', zip_path, Path(zip_path).exists())
print('BUNDLE_ITEMS=', sorted([p.name for p in bundle_root.iterdir()]))

BUNDLE_DIR= /content/seg17c_artifacts
BUNDLE_ZIP= /content/seg17c_artifacts_bundle.zip True
BUNDLE_ITEMS= ['README.txt', 'metrics.json', 'yolo26n_seg_17c_best.onnx', 'yolo26n_seg_17c_best.pt', 'yolov8n_seg_17c_best.onnx', 'yolov8n_seg_17c_best.pt']


In [36]:
# 7.12) Set HF token in runtime (no notebook save)
import os

token = input('HF_TOKEN: ').strip()
assert token, 'Empty token.'
os.environ['HF_TOKEN'] = token
del token
print('HF_TOKEN loaded into runtime environment.')

HF_TOKEN loaded into runtime environment.


In [37]:
# 7.13) Validate Hugging Face token
import os
from huggingface_hub import HfApi

HF_TOKEN = os.getenv('HF_TOKEN', '').strip()
assert HF_TOKEN, 'HF_TOKEN is empty. Run cell "# 7.12) Set HF token in runtime (no notebook save)" first.'

who = HfApi(token=HF_TOKEN).whoami()
print('HF auth OK as:', who.get('name', '<unknown>'))

HF auth OK as: Jpjpcs


In [38]:
# 7.15) Configure HF repo settings (token from runtime env)
import os

# Non-blocking setup for VS Code notebook integration.
existing_token = os.getenv('HF_TOKEN', '').strip()
if existing_token:
    print('HF_TOKEN already set in runtime environment.')
else:
    print('HF_TOKEN is not set yet (no value saved in notebook).')
    print('Run this in a temporary cell, then rerun this cell:')
    print('import os; os.environ["HF_TOKEN"] = input("HF_TOKEN: ").strip()')

# Replace with your real owner/repo, e.g. Jpjpcs/sea-me-artifacts or org-name/sea-me-artifacts
os.environ['HF_REPO_ID'] = 'Jpjpcs/sea-me-artifacts'
os.environ['HF_REPO_TYPE'] = 'dataset'

print('HF_REPO_ID=', os.environ['HF_REPO_ID'])
print('HF_REPO_TYPE=', os.environ['HF_REPO_TYPE'])

HF_TOKEN already set in runtime environment.
HF_REPO_ID= Jpjpcs/sea-me-artifacts
HF_REPO_TYPE= dataset


In [39]:
# 7.2) Non-interactive upload to Hugging Face Hub (optional, recommended for VSCode automation)
import os
from pathlib import Path
from datetime import datetime, timezone

try:
    from huggingface_hub import HfApi
    from huggingface_hub.errors import HfHubHTTPError
except Exception:
    !pip -q install huggingface_hub
    from huggingface_hub import HfApi
    from huggingface_hub.errors import HfHubHTTPError

BUNDLE_ZIP = '/content/seg17c_artifacts_bundle.zip'
assert Path(BUNDLE_ZIP).exists(), f'Missing bundle: {BUNDLE_ZIP}. Run cell titled "# 7.1) Bundle trained artifacts into a single zip for download" first.'

HF_TOKEN = os.getenv('HF_TOKEN', '').strip()
HF_REPO_ID = os.getenv('HF_REPO_ID', '').strip()  # e.g. your-user/sea-me-artifacts
HF_REPO_TYPE = os.getenv('HF_REPO_TYPE', 'dataset').strip()  # dataset or model

if (
    not HF_TOKEN
    or HF_TOKEN.lower() in {'hf_xxx', 'your_hf_token'}
    or not HF_REPO_ID
    or '/' not in HF_REPO_ID
):
    print('HF upload skipped: invalid/missing env vars.')
    print('Set env vars and rerun this cell:')
    print('  import os; os.environ["HF_TOKEN"] = input("HF_TOKEN: ").strip()')
    print('  os.environ["HF_REPO_ID"] = "your-user/sea-me-artifacts"')
    print('  os.environ["HF_REPO_TYPE"] = "dataset"')
else:
    api = HfApi(token=HF_TOKEN)
    try:
        who = api.whoami()
        print('HF auth OK as:', who.get('name', '<unknown>'))

        ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
        remote_path = f'seg17c/{ts}/seg17c_artifacts_bundle.zip'

        # Create repo if missing.
        api.create_repo(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, exist_ok=True)

        api.upload_file(
            path_or_fileobj=BUNDLE_ZIP,
            path_in_repo=remote_path,
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
        )

        print('HF upload OK')
        print('HF_REMOTE_PATH=', remote_path)
        print('LENOVO_PULL_CMD_HF=')
        print(
            f"hf download {HF_REPO_ID} {remote_path} "
            f"--type {HF_REPO_TYPE} --local-dir ~/Documents/AI/Yolo_benchmark/trained/seg17c_bundle"
        )
        print('LENOVO_FETCH_SCRIPT_CMD=')
        print(
            f"HF_REPO_ID={HF_REPO_ID} HF_REMOTE_PATH={remote_path} HF_REPO_TYPE={HF_REPO_TYPE} "
            "bash src/hailo/scripts/fetch_seg17c_artifacts.sh"
        )
    except HfHubHTTPError as exc:
        status = getattr(getattr(exc, 'response', None), 'status_code', None)
        print(f'HF upload failed (HTTP {status}).')
        if status == 401:
            print('Token invalid/expired. Generate a NEW User Access Token with WRITE permission.')
            print('Then rerun only these cells in order: Cell 17 -> Cell 18 -> Cell 20.')
            print('Do not continue to ONNX/HEF steps before upload succeeds.')
        elif status == 403:
            print('Token authenticated but lacks permission for this repo. Check repo ownership and token scope.')
        else:
            print('Hub request failed. Error:', exc)

HF auth OK as: Jpjpcs


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...g17c_artifacts_bundle.zip:  99%|#########8| 32.6MB / 33.0MB            

HF upload OK
HF_REMOTE_PATH= seg17c/20260428_000224/seg17c_artifacts_bundle.zip
LENOVO_PULL_CMD_HF=
hf download Jpjpcs/sea-me-artifacts seg17c/20260428_000224/seg17c_artifacts_bundle.zip --type dataset --local-dir ~/Documents/AI/Yolo_benchmark/trained/seg17c_bundle
LENOVO_FETCH_SCRIPT_CMD=
HF_REPO_ID=Jpjpcs/sea-me-artifacts HF_REMOTE_PATH=seg17c/20260428_000224/seg17c_artifacts_bundle.zip HF_REPO_TYPE=dataset bash src/hailo/scripts/fetch_seg17c_artifacts.sh


In [ ]:
# 8) Export trained models to ONNX
from pathlib import Path
from ultralytics import YOLO

# Guard rail for the required process-mae order.
if not globals().get('VASCO_COMPARISON_OK', False):
    raise RuntimeError(
        'Stop: complete Step 3 (comparison with YOLOv8s Vasco) first. '
        'When done, set VASCO_COMPARISON_OK = True in a new cell and rerun this cell.'
    )

EXPORT_IMGSZ_8N = 640
EXPORT_IMGSZ_26N = 640

exported = {}
for model_name, best_path, export_imgsz in [
    ('yolov8n_seg_17c', BEST_8N, EXPORT_IMGSZ_8N),
    ('yolo26n_seg_17c', BEST_26N, EXPORT_IMGSZ_26N),
]:
    model = YOLO(best_path)
    out = model.export(format='onnx', imgsz=export_imgsz, simplify=True, opset=13)
    exported[model_name] = out
    print(model_name, 'onnx=', out, Path(out).exists())

print('Exported ONNX artifacts:', exported)

Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLOv8n-seg summary (fused): 86 layers, 3,261,379 parameters, 0 gradients, 11.4 GFLOPs

PyTorch: starting from '/content/seame_seg_17c/yolov8n_seg_17c/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 53, 8400), (1, 32, 160, 160)) (6.5 MB)

ONNX: starting export with onnx 1.21.0 opset 13...
ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 1.2s, saved as '/content/seame_seg_17c/yolov8n_seg_17c/weights/best.onnx' (12.7 MB)

Export complete (1.6s)
Results saved to /content/seame_seg_17c/yolov8n_seg_17c/weights
Predict:         yolo predict task=segment model=/content/seame_seg_17c/yolov8n_seg_17c/weights/best.onnx imgsz=640 
Validate:        yolo val task=segment model=/content/seame_seg_17c/yolov8n_seg_17c/weights/best.onnx imgsz=640 data=/content/dataset_17c/data.yaml  
Visualize:       https://netron.app
yolov8n_seg_17c onnx= /content/seame_seg_17c/yolov8n_s